**Load Data**

In [4]:
# Step 1: Upload and Load CSV file
from google.colab import files
import pandas as pd

# Upload file from your local machine
uploaded = files.upload()  # This opens a file picker — select your CSV (e.g., spam.csv)

# Load into DataFrame (replace 'spam.csv' with your actual filename if different)
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only relevant columns (v1 = label, v2 = message)
df = df[['v1', 'v2']]
df.columns = ['label', 'message']  # Rename for clarity

# Show first few rows
df.head()

Saving spam.csv to spam (1).csv


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


**Analyze Data — Head, dtypes, Nulls, Duplicates**

In [5]:
# Step 2: Basic Data Analysis

print("=== Shape ===")
print(df.shape)

print("\n=== Column Types ===")
print(df.dtypes)

print("\n=== Null Values ===")
print(df.isnull().sum())

print("\n=== Label Distribution ===")
print(df['label'].value_counts())

print("\n=== Duplicates ===")
print(f"Number of duplicate messages: {df.duplicated(subset='message').sum()}")

# Optional: Show a few duplicate examples
print("\n=== Sample Duplicates ===")
print(df[df.duplicated(subset='message', keep=False)].head(6))

=== Shape ===
(5572, 2)

=== Column Types ===
label      object
message    object
dtype: object

=== Null Values ===
label      0
message    0
dtype: int64

=== Label Distribution ===
label
ham     4825
spam     747
Name: count, dtype: int64

=== Duplicates ===
Number of duplicate messages: 403

=== Sample Duplicates ===
   label                                            message
2   spam  Free entry in 2 a wkly comp to win FA Cup fina...
7    ham  As per your request 'Melle Melle (Oru Minnamin...
8   spam  WINNER!! As a valued network customer you have...
9   spam  Had your mobile 11 months or more? U R entitle...
11  spam  SIX chances to win CASH! From 100 to 20,000 po...
12  spam  URGENT! You have won a 1 week FREE membership ...


Data Cleaning

In [6]:
# Step 3: Clean the message text
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing spaces
    text = text.strip()
    return text

# Apply cleaning
df['message_clean'] = df['message'].apply(clean_text)

# Remove duplicate messages (keep first)
df = df.drop_duplicates(subset='message_clean', keep='first').reset_index(drop=True)

print(f"Shape after removing duplicates: {df.shape}")

Shape after removing duplicates: (5157, 3)


In [7]:
# Step 4: Convert labels to numeric (ham=0, spam=1)
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

# Verify
print(df[['label', 'label_encoded']].head())

  label  label_encoded
0   ham              0
1   ham              0
2  spam              1
3   ham              0
4   ham              0


**Train/Validation Split**

In [8]:
from sklearn.model_selection import train_test_split

X = df['message_clean'].values
y = df['label_encoded'].values

# Keep 20% for validation — stratified to preserve spam ratio
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {len(X_train)} samples ({y_train.mean():.1%} spam)")
print(f"Val:   {len(X_val)} samples ({y_val.mean():.1%} spam)")

Train: 4125 samples (12.5% spam)
Val:   1032 samples (12.4% spam)


**Tokenize & Pad Sequences (for LSTM)**

In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Parameters
MAX_WORDS = 8000    # Vocabulary size (covers ~99% of SMS words)
MAX_LEN = 50        # Max SMS length (most are < 20 words)

# Initialize and fit tokenizer on training data only
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

# Pad sequences to same length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Training sequences shape: {X_train_pad.shape}")
print(f"Validation sequences shape: {X_val_pad.shape}")

Vocabulary size: 7830
Training sequences shape: (4125, 50)
Validation sequences shape: (1032, 50)


Build & Train LSTM Model

In [10]:
# Build the model
model = tf.keras.Sequential([
    # Embedding layer: maps word indices to dense vectors
    tf.keras.layers.Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LEN,
        name='embedding'
    ),

    # Regularization
    tf.keras.layers.SpatialDropout1D(0.3),

    # LSTM layer with dropout
    tf.keras.layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3, name='lstm'),

    # Dense layers
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    # Output layer (sigmoid for binary classification)
    tf.keras.layers.Dense(1, activation='sigmoid', name='output')
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model structure
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Compute Class Weights & Train**

In [12]:
# Add precision and recall metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Compute class weights (unchanged)
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print("Class weights → Ham:", class_weights[0], "| Spam:", class_weights[1])

# ✅ UPDATED CALLBACKS: Monitor VAL_RECALL for early stopping
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_recall',      # ← Key change: stop when spam recall stops improving
        mode='max',                # We want to MAXIMIZE recall
        patience=4,                # Give more room (recall can fluctuate)
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train with updated metrics & callbacks
print("\n🚀 Starting training with Precision/Recall monitoring...\n")
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=25,                # Slightly more epochs (recall may improve slower)
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

Class weights → Ham: 0.5711714206590972 | Spam: 4.012645914396887

🚀 Starting training with Precision/Recall monitoring...

Epoch 1/25
129/129 ━━━━━━━━━━━━━━━━━━━━ 28s 187ms/step - accuracy: 0.9854 - loss: 0.1738 - precision: 0.9573 - recall: 0.9247 - val_accuracy: 0.9777 - val_loss: 0.1427 - val_precision: 0.9412 - val_recall: 0.8750 - learning_rate: 0.0010
Epoch 2/25
129/129 ━━━━━━━━━━━━━━━━━━━━ 24s 183ms/step - accuracy: 0.9917 - loss: 0.1233 - precision: 0.9849 - recall: 0.9496 - val_accuracy: 0.9729 - val_loss: 0.1528 - val_precision: 0.9032 - val_recall: 0.8750 - learning_rate: 0.0010
Epoch 3/25
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.9908 - loss: 0.0981 - precision: 0.9555 - recall: 0.9723
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
129/129 ━━━━━━━━━━━━━━━━━━━━ 24s 184ms/step - accuracy: 0.9908 - loss: 0.0982 - precision: 0.9555 - recall: 0.9722 - val_accuracy: 0.9777 - val_loss: 0.1467 - val_precision: 0.8832 - val_recall: 0.9453

In [13]:
# Evaluate full metrics
val_loss, val_acc, val_prec, val_rec = model.evaluate(X_val_pad, y_val, verbose=0)

# Compute F1-Score manually (not built-in in older TF)
val_f1 = 2 * (val_prec * val_rec) / (val_prec + val_rec) if (val_prec + val_rec) > 0 else 0

print("\n" + "="*50)
print("✅ FINAL VALIDATION METRICS")
print("="*50)
print(f"Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"Precision: {val_prec:.4f} ({val_prec*100:.2f}%) → % of predicted spam that is truly spam")
print(f"Recall   : {val_rec:.4f} ({val_rec*100:.2f}%) → % of actual spam correctly caught")
print(f"F1-Score : {val_f1:.4f} ({val_f1*100:.2f}%)")
print("="*50)


✅ FINAL VALIDATION METRICS
Accuracy : 0.9777 (97.77%)
Precision: 0.8832 (88.32%) → % of predicted spam that is truly spam
Recall   : 0.9453 (94.53%) → % of actual spam correctly caught
F1-Score : 0.9132 (91.32%)


In [14]:
import pickle
import os

# Create a directory to save everything
model_dir = "sms_spam_lstm_model"
os.makedirs(model_dir, exist_ok=True)

# 1. Save the Keras model (architecture + weights)
model.save(os.path.join(model_dir, "sms_spam_model.h5"))

# 2. Save the tokenizer (so you can preprocess new SMS later)
with open(os.path.join(model_dir, "tokenizer.pickle"), "wb") as f:
    pickle.dump(tokenizer, f)

# 3. Save key parameters (MAX_LEN, MAX_WORDS) for inference
import json
config = {
    "MAX_LEN": MAX_LEN,
    "MAX_WORDS": MAX_WORDS
}
with open(os.path.join(model_dir, "model_config.json"), "w") as f:
    json.dump(config, f)

print("✅ Model, tokenizer, and config saved to:", model_dir)

✅ Model, tokenizer, and config saved to: sms_spam_lstm_model


In [15]:
# Zip the folder for easy download
!zip -r sms_spam_lstm_model.zip sms_spam_lstm_model

# Trigger download in Colab
from google.colab import files
files.download("sms_spam_lstm_model.zip")

  adding: sms_spam_lstm_model/ (stored 0%)
  adding: sms_spam_lstm_model/model_config.json (deflated 9%)
  adding: sms_spam_lstm_model/tokenizer.pickle (deflated 52%)
  adding: sms_spam_lstm_model/sms_spam_model.h5 (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>